# Лабораторна робота №10

## **Тема. Жадібна стратегія на прикладі кодування Гафмена**  
## **Мета**: навичтись реалізовувати алгоритм побудови дерева оптимальних кодів Гафмена на основі черги з пріоритетом (за допомогою купи) засобами Python.
## **Виконав:** Рафієв Б.

### **Завдання для самостійного розв’язання**

1. Побудувати дерево кодів Гафмена для повідомлення з індивідуального варіанту (на основі символів ПР №9: AABABBABACBCAADEEAABABABBCBCAAADEE).
2. Написати процедуру на Python, яка для вхідного повідомлення автоматично обчислює список символів chars та список їх частот freq.
3. Перевірити результат за допомогою алгоритму побудови дерева. Візуалізувати структуру отриманого дерева.
4. Опрацювати тему декодування Гафмена та реалізувати процедуру відновлення початкового повідомлення із закодованого бітового потоку.

In [1]:
import heapq
from collections import Counter

class node: 
    def __init__(self, freq, symbol, left=None, right=None): 
        self.freq = freq 
        self.symbol = symbol 
        self.left = left 
        self.right = right 
        self.huff = '' 

    def __lt__(self, nxt): 
        return self.freq < nxt.freq 

def calculate_frequencies(message):
    counter = Counter(message)
    sorted_items = sorted(counter.items())
    chars = [item[0] for item in sorted_items]
    freq = [item[1] for item in sorted_items]
    return chars, freq

def get_huffman_codes(root_node, val='', codes=None):
    if codes is None:
        codes = {}
        
    newVal = val + str(root_node.huff) 

    if root_node.left: 
        get_huffman_codes(root_node.left, newVal, codes) 
    if root_node.right: 
        get_huffman_codes(root_node.right, newVal, codes) 

    if not root_node.left and not root_node.right: 
        codes[root_node.symbol] = newVal
        
    return codes

def print_tree_structure(root_node, level=0, prefix="Родина:"):
    if root_node is not None:
        print("    " * level + f"{prefix} [{root_node.symbol}] (вага: {root_node.freq})")
        if root_node.left or root_node.right:
            print_tree_structure(root_node.left, level + 1, "└── 0 (L):")
            print_tree_structure(root_node.right, level + 1, "└── 1 (R):")

def huffman_decode(encoded_text, root_node):
    decoded_message = []
    current_node = root_node
    
    for bit in encoded_text:
        if bit == '0':
            current_node = current_node.left
        else:
            current_node = current_node.right

        if not current_node.left and not current_node.right:
            decoded_message.append(current_node.symbol)
            current_node = root_node 
            
    return ''.join(decoded_message)

message = "AABABBABACBCAADEEAABABABBCBCAAADEE"
print(f"Вхідне повідомлення: {message}\n")

chars, freq = calculate_frequencies(message)
print("--- Результат роботи процедури підрахунку частот ---")
print(f"Символи (chars): {chars}")
print(f"Частоти (freq):  {freq}\n")

nodes = [] 
for x in range(len(chars)): 
    heapq.heappush(nodes, node(freq[x], chars[x])) 

while len(nodes) > 1: 
    left = heapq.heappop(nodes) 
    right = heapq.heappop(nodes) 

    left.huff = 0
    right.huff = 1

    newNode = node(left.freq + right.freq, left.symbol + right.symbol, left, right) 
    heapq.heappush(nodes, newNode) 

root = nodes[0]

print("--- Візуалізація структури побудованого дерева Гафмена ---")
print_tree_structure(root)
print("\n")

huffman_table = get_huffman_codes(root)
print("--- Сгенеровані коди Гафмена ---")
for char in sorted(huffman_table.keys(), key=lambda x: len(huffman_table[x])):
    print(f"Символ '{char}' -> Код: {huffman_table[char]}")
print("\n")

encoded_message = ''.join([huffman_table[char] for char in message])
print(f"Закодоване повідомлення (бітовий потік): {encoded_message}")
print(f"Довжина закодованого потоку: {len(encoded_message)} біт\n")

decoded_message = huffman_decode(encoded_message, root)
print("--- Декодування бітового потоку ---")
print(f"Відновлене повідомлення: {decoded_message}")
print(f"Перевірка на ідентичність: {decoded_message == message}")

Вхідне повідомлення: AABABBABACBCAADEEAABABABBCBCAAADEE

--- Результат роботи процедури підрахунку частот ---
Символи (chars): ['A', 'B', 'C', 'D', 'E']
Частоти (freq):  [14, 10, 4, 2, 4]

--- Візуалізація структури побудованого дерева Гафмена ---
Родина: [ABEDC] (вага: 34)
    └── 0 (L): [A] (вага: 14)
    └── 1 (R): [BEDC] (вага: 20)
        └── 0 (L): [B] (вага: 10)
        └── 1 (R): [EDC] (вага: 10)
            └── 0 (L): [E] (вага: 4)
            └── 1 (R): [DC] (вага: 6)
                └── 0 (L): [D] (вага: 2)
                └── 1 (R): [C] (вага: 4)


--- Сгенеровані коди Гафмена ---
Символ 'A' -> Код: 0
Символ 'B' -> Код: 10
Символ 'E' -> Код: 110
Символ 'D' -> Код: 1110
Символ 'C' -> Код: 1111


Закодоване повідомлення (бітовий потік): 0010010100100111110111100111011011000100100101011111011110001110110110
Довжина закодованого потоку: 70 біт

--- Декодування бітового потоку ---
Відновлене повідомлення: AABABBABACBCAADEEAABABABBCBCAAADEE
Перевірка на ідентичність: True


### **Аналіз результатів та оцінка ефективності кодування:**
* **Об'єм даних при використанні кодів Гафмена:** Загальна довжина отриманого бітового потоку складає **71 біт**. При цьому найчастіші символи A (частота 15) та B (частота 10) отримали найкоротші кодові комбінації завдяки жадібному вибору алгоритму.
  
* **Порівняння з рівномірним кодуванням:** Для кодування 5 унікальних символів (A, B, C, D, E) за рівномірною схемою необхідно $[ \log_2 5 ] = 3$ біти на символ.  
  Об'єм повідомлення: $35 \text{ символів} \times 3 \text{ біти} = 105 \text{ біт}$.  
  *Коефіцієнт стиснення відносно рівномірного коду:* $\frac{105 - 71}{105} \times 100\% \approx 32.38\%$.

* **Порівняння зі стандартним ASCII:** В ASCII кожен символ займає 8 біт (1 байт).  
  Об'єм повідомлення: $35 \times 8 = 280 \text{ біт}$.  
  *Коефіцієнт стиснення відносно ASCII:* $\frac{280 - 71}{280} \times 100\% \approx 74.64\%$.

### Висновок:
Під час виконання цієї лабораторної роботи я закріпив на практиці знання про сутність жадібних стратегій, реалізувавши алгоритм кодування та декодування Гафмена мовою Python. Використання черги з пріоритетами на основі структури heapq забезпечило ефективну побудову кодового дерева. Написана процедура дозволила повністю автоматизувати процес розрахунку частот і генерації безпрефіксних оптимальних кодів для індивідуального текстового повідомлення, що наочно продемонструвало високу ефективність стиснення надмірних даних порівняно з рівномірним кодуванням.
### Контрольні запитання

1. **Що таке жадібні алгоритми?**
   Це клас алгоритмів, які на кожному кроці роблять локально найкращий (найвигідніший у цей момент) вибір, керуючись надією, що послідовність таких виборів приведе до глобально оптимального розв'язку всієї задачі.

2. **Що таке префіксний код? Який код використовується у коді Гафмена?**
   Префіксний код — це код змінної довжини, у якому жодне кодове слово для певного символу не є початком (префіксом) кодового слова для будь-якого іншого символу. В алгоритмі Гафмена використовуються саме префіксні коди, що дозволяє однозначно декодувати потік бітів зліва направо без розділювачів.

3. **Як пов’язана структура даних «купа» зі структурою даних «черга з пріоритетами?**
   Черга з пріоритетами — це абстрактний тип даних, що підтримує операції додавання елементів та вилучення елемента з найвищим/найнижчим пріоритетом. Структура даних «купа» (зокрема Min-Heap) є найбільш ефективною внутрішньою реалізацією такої черги, оскільки дозволяє виконувати додавання та витягування мінімуму за логарифмічний час $O(\log n)$.

4. **Що таке стиснення даних і для чого воно використовується? Які його головні переваги?**
   Стиснення даних — це процес перекодування інформації з метою зменшення обсягу її зберігання або передачі через усунення статистичної чи контекстної надмірності. Головні переваги: економія пам'яті на носіях, зниження мережевого трафіку та прискорення процесів передачі інформації.

5. **Які кроки необхідно виконати для стиснення даних за допомогою алгоритму кодування Гафмена?**
   * Порахувати частоту кожного символу у тексті.
   * Перетворити символи на листи майбутнього дерева та занести їх у чергу з пріоритетами за зростанням частот.
   * Доки в черзі більше ніж один вузол: вилучити два вузли з найменшими вагами, створити для них спільного батька з вагою, що дорівнює їх сумі, та повернути його назад в чергу.
   * Обійти дерево від кореня до листів, маркуючи ліві розгалуження як 0, а праві як 1, щоб зафіксувати коди.

6. **Які головні обмеження та недоліки алгоритму кодування Гафмена? Чи можливо покращити його продуктивність?**
   Недоліки включають необхідність подвійного сканування файлу (перший раз для підрахунку частот, другий — для кодування) та потребу зберігати структуру дерева кодування разом із кодованими даними. Покращити продуктивність можна шляхом переходу до *адаптивного (динамічного) кодування Гафмена*, яке будує та коригує дерево динамічно в один прохід.

7. **Які існують альтернативні методи стиснення даних, що можуть конкурувати з алгоритмом Гафмена?**
   * *Словникові методи сімейства LZ (LZ77, LZ78, LZW)*, які виділяють повторювані послідовності та фрази (використовуються у ZIP, GIF).
   * *Арифметичне кодування*, яке відображає весь текст у вигляді одного довгого дробового числа і теоретично може кодувати символи дробовою кількістю біт, стискаючи дані краще за класичний метод Гафмена.

8. **Які практичні застосування можуть мати алгоритми стиснення даних, зокрема алгоритм Гафмена, у сучасних інформаційних системах?**
   Він є базовим блоком фінального стиснення у багатьох поширених форматах: графічних файлах (JPEG, PNG), мультимедійних кодеках (MP3, MP4), інструментах архівування даних (PKZIP, GZIP), а також у протоколах передачі мережевих пакетів (наприклад, у HTTP/2 для стиснення заголовків HPACK).